In [4]:
import openmc 
from openmc.deplete import d1s
from openmc.data import effective_dose
from openmc_fusion_benchmarks.neutron_sources import fng_source

model = openmc.Model()

# Define the materials
material = openmc.Materials.from_xml('materials.xml')
# material.cross_sections =  '/home/grego01/Nuclear_data/endfb-viii.0-hdf5/cross_sections.xml'
model.materials = material

# Define the geometry
geom = openmc.Geometry.from_xml('geometry.xml')
model.geometry = geom

# Define Settings
# source definition
fng_center = (0, 0, 0)
fng_uvw = (0., 1., 0)

source = fng_source(center=fng_center,
                    reference_uvw=fng_uvw, beam_energy=230)

# settings
settings = openmc.Settings(run_mode='fixed source')
settings.batches = 100
settings.particles = 1000
settings.source = source
settings.photon_transport = True
settings.verbosity = 5
settings.output = {'tallies': False}

model.settings = settings

############################################################################
# Specify Tallies
model.tallies = openmc.Tallies()

# particle filters
neutron_filter = openmc.ParticleFilter(['neutron'])
photon_filter = openmc.ParticleFilter(['photon'])

# cell filters
cell_filter_9 = openmc.CellFilter([621, 622, 624, 625])
cell_filter_10 = openmc.CellFilter([178, 179, 180, 181, 182, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 636, 637, 638])
cell_filter_photon = openmc.CellFilter([227])

# Energy filters
energies, coeffs = effective_dose.dose.dose_coefficients('photon', data_source='icrp74')
dose_filter = openmc.EnergyFunctionFilter(energies, coeffs)

tally = openmc.Tally(name="sdr_dose")
tally.filters = [cell_filter_photon, photon_filter, dose_filter]
tally.scores = ['flux']
model.tallies.append(tally)

tally = openmc.Tally(name="foil_activation")
tally.filters = [cell_filter_10, neutron_filter]
tally.scores = ['(n,gamma)', '(n,p)', '(n,2n)', '(n,3n)']
model.tallies.append(tally)

tally = openmc.Tally(name="neutron_flux")
tally.filters = [cell_filter_photon, neutron_filter]
tally.scores = ['flux']
model.tallies.append(tally)

model.export_to_xml()

# openmc.config['chain_file'] = r'/home/davide/openmc_models/CROSS_SECTIONS/decay/endfb-viii.0-chain/chain_endfb80_sfr.xml'
# model.settings.photon_transport = True
# model.settings.use_decay_photons = True
# d1s.prepare_tallies(model)

/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=1.
  warn(msg, IDWarning)
/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=2.
  warn(msg, IDWarning)
/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=3.
  warn(msg, IDWarning)
/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=4.
  warn(msg, IDWarning)
/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material instance already exists with id=5.
  warn(msg, IDWarning)
/home/davide/anaconda3/envs/openmc0.15.1/lib/python3.13/site-packages/openmc/mixin.py:70: IDWarning: Another Material in

In [5]:
sp = openmc.StatePoint('statepoint.100.h5')

# only for d1s
# d1s depletion
# model.settings.use_decay_photons = True

# nuclides = d1s.prepare_tallies(model)

# time_steps = [19440, 61680, 32940, 54840, 15720, 6360, 8940]
# source_rates = [2.32e10, 0.0, 2.87e10, 0.0, 1.90e10, 0.0, 1.36e10]

# factors = d1s.time_correction_factors(nuclides, time_steps, source_rates)
tally_dose = sp.get_tally(name='sdr_dose')
tally_rr = sp.get_tally(name='foil_activation')
tally_neutron = sp.get_tally(name='neutron_flux')

print(tally_neutron.get_values())
# tally_dose = d1s.apply_time_correction(tally_dose, factors)
print(tally_dose.get_values())
print(tally_rr.get_values())

[[[0.26445368]]]
[[[0.00000000e+00]]

 [[1.13422544e-07]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[1.02257211e-05]]

 [[5.31206245e-05]]

 [[1.01476200e-20]]

 [[4.84544833e-06]]

 [[1.91446189e-04]]

 [[0.00000000e+00]]

 [[5.85722089e-04]]

 [[3.45630391e-07]]

 [[1.13951369e-03]]

 [[0.00000000e+00]]

 [[3.38265166e-08]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[1.01113323e-06]]

 [[0.00000000e+00]]

 [[3.32079424e-07]]

 [[0.00000000e+00]]

 [[5.62272818e-05]]

 [[7.89819228e-04]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[6.72400842e-07]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00]]

 [[0.00000000e+00